in the name of Allah
# Fraud Detection in Financial Transactions and Transaction Classification (based on Fraud) using a Machine Learning-based system
### Electronic Commerce final project - Winter 2026
------------------

In [3]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, fbeta_score
)

## TASK I: Exploratory Data Analysis (EDA) and Data Preprocessing

In this task, we are responsible for loading, inspecting, and cleaning the raw transaction dataset, followed by extracting key statistical insights. This includes the following steps:

1. **Load and inspect the dataset**: The dataset is read from `fraud_detection_2.csv`. Initial properties such as row count, column count, and column names are reported to confirm correct loading.

2. **Handle NULL values and duplicate rows**: All columns are checked for missing values. If any exist, the corresponding rows are dropped. Complete duplicate rows are identified and removed to ensure data integrity.

3. **Fraud count analysis per account**: Transactions are grouped by account ID and the total number of fraudulent records (`Class = 1`) per account is computed. The accounts with the highest and lowest (non-zero) fraud counts are identified.

4. **Amount statistics for fraudulent transactions**: All records labeled as fraud are filtered and descriptive statistics — mean, median, and standard deviation of the `Amount` column — are computed to characterize the financial profile of fraudulent activity.

5. **Balance analysis (last record per account)**: For each account, only the most recent transaction record is retained. The accounts with the highest and lowest transaction amounts among these final records are identified.

6. **Overall dataset summary and save cleaned data**: Key aggregate statistics are reported, including total unique accounts, total records, fraud vs. normal record counts, fraud percentage, and the class imbalance ratio (Normal/Fraud). The cleaned dataset is saved to `ResultFiles/1_cleaned_fraud_data.csv` for use in subsequent tasks.

In [4]:
# ----------------------------------------------------------------------------
# 1. Load & Initial Data Inspection
# ----------------------------------------------------------------------------
print("="*80)
print("TASK I: EXPLORATORY DATA ANALYSIS")
print("="*80)

df = pd.read_csv('fraud_detection_2.csv')

print("\n[1] Initial Data Overview:")
print(f"    Rows: {df.shape[0]:,}")
print(f"    Columns: {df.shape[1]}")
print(f"    Column names: {list(df.columns)}")

TASK I: EXPLORATORY DATA ANALYSIS

[1] Initial Data Overview:
    Rows: 100,491
    Columns: 31
    Column names: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


In [5]:
# ----------------------------------------------------------------------------
# 2. Check for NULL Values & Complete Duplicate Rows
# ----------------------------------------------------------------------------
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()

before_dup = len(df)
df.drop_duplicates(inplace=True)
duplicates_removed = before_dup - len(df)

if total_nulls > 0:
    initial_rows = len(df)
    df.dropna(inplace=True)
    print(f"    Rows removed: {initial_rows - len(df)}")
else:
    print("    No NULL values found ✓")

if duplicates_removed > 0:
    print(f"    Duplicate rows removed: {duplicates_removed}")
else:
    print("    No duplicate rows found ✓")
print(f"[2] Dataset size after cleaning: {len(df):,} rows")

    No NULL values found ✓
    Duplicate rows removed: 623
[2] Dataset size after cleaning: 99,868 rows


In [6]:
# ----------------------------------------------------------------------------
# 3. Fraud Count Analysis per Account (ALL RECORDS)
# ----------------------------------------------------------------------------
print("\n[3] Fraud Count Analysis (per account):")

# Calculate total fraud count for each account across ALL their records
fraud_count = df.groupby('id')['Class'].sum()

# Maximum fraud count
max_fraud_id = fraud_count.idxmax()
max_fraud_count = int(fraud_count.max())
print(f"    Account with MOST fraud: ID={max_fraud_id} | Count={max_fraud_count}")

# Minimum fraud count (excluding zero)
fraud_non_zero = fraud_count[fraud_count > 0]
if len(fraud_non_zero) > 0:
    min_fraud_id = fraud_non_zero.idxmin()
    min_fraud_count = int(fraud_non_zero.min())
    print(f"    Account with LEAST fraud (non-zero): ID={min_fraud_id} | Count={min_fraud_count}")
else:
    print("    No fraudulent accounts found!")


[3] Fraud Count Analysis (per account):
    Account with MOST fraud: ID=84204.0 | Count=2
    Account with LEAST fraud (non-zero): ID=406.0 | Count=1


In [7]:
# ----------------------------------------------------------------------------
# 4. Amount Statistics for Fraudulent Transactions (ALL CLASS=1 RECORDS)
# ----------------------------------------------------------------------------
print("\n[4] Amount Statistics for Fraudulent Transactions:")

# Filter ALL fraudulent transactions
fraud_amounts = df[df['Class'] == 1]['Amount']

if len(fraud_amounts) > 0:
    mean_fraud = fraud_amounts.mean()
    median_fraud = fraud_amounts.median()
    std_fraud = fraud_amounts.std()
    
    print(f"    Mean:   {mean_fraud:>10.2f}")
    print(f"    Median: {median_fraud:>10.2f}")
    print(f"    Std:    {std_fraud:>10.2f}")
else:
    print("    No fraudulent transactions found!")


[4] Amount Statistics for Fraudulent Transactions:
    Mean:       123.87
    Median:       9.82
    Std:        260.21


In [8]:
# ----------------------------------------------------------------------------
# 5. Balance Analysis (ONLY LAST RECORD PER ACCOUNT)
# ----------------------------------------------------------------------------
print("\n[5] Balance Analysis:")

# Get the LAST record for each account (most recent balance)
df_last = df.groupby('id').tail(1).reset_index(drop=True)

# Maximum balance
max_balance_idx = df_last['Amount'].idxmax()
max_balance_id = df_last.loc[max_balance_idx, 'id']
max_balance = df_last.loc[max_balance_idx, 'Amount']
print(f"    HIGHEST balance: ID={max_balance_id} | Amount={max_balance:>10.2f}")

# Minimum balance
min_balance_idx = df_last['Amount'].idxmin()
min_balance_id = df_last.loc[min_balance_idx, 'id']
min_balance = df_last.loc[min_balance_idx, 'Amount']
print(f"    LOWEST balance:  ID={min_balance_id} | Amount={min_balance:>10.2f}")


[5] Balance Analysis:
    HIGHEST balance: ID=42951.0 | Amount=  12910.93
    LOWEST balance:  ID=804.0 | Amount=      0.00


In [9]:
# ----------------------------------------------------------------------------
# 6. Overall Dataset Summary & Save Cleaned Data
# ----------------------------------------------------------------------------
print("\n[6] Overall Dataset Summary:")
print(f"    Total unique accounts: {df['id'].nunique():,}")
print(f"    Total transaction records: {len(df):,}")
print(f"    Fraudulent records: {(df['Class'] == 1).sum():,}")
print(f"    Normal records: {(df['Class'] == 0).sum():,}")

print("    ----------------------------------")
fraud_percentage = (df['Class'] == 1).sum() / len(df) * 100
print(f"    Fraud percentage: {fraud_percentage:.2f}%")
print(f"    Imbalance ratio: {((df['Class'] == 0).sum() / (df['Class'] == 1).sum()):.2f} (Normal/Fraud)")

# Create output directory if it doesn't exist & save cleaned data
task2_output_dir = "ResultFiles"
os.makedirs(task2_output_dir, exist_ok=True)
output_file = os.path.join(task2_output_dir, '1_cleaned_fraud_data.csv')
df.to_csv(output_file, index=False)
print(f"    ✓ Cleaned data saved to: {output_file}")

print("\n" + "="*80)
print("TASK I COMPLETED SUCCESSFULLY")
print("="*80)


[6] Overall Dataset Summary:
    Total unique accounts: 45,792
    Total transaction records: 99,868
    Fraudulent records: 473
    Normal records: 99,395
    ----------------------------------
    Fraud percentage: 0.47%
    Imbalance ratio: 210.14 (Normal/Fraud)
    ✓ Cleaned data saved to: ResultFiles\1_cleaned_fraud_data.csv

TASK I COMPLETED SUCCESSFULLY


## TASK II: Data Visualization and Exploratory Graphical Analysis

In this task, we focus on creating comprehensive visualizations to understand feature distributions, class separability, and fraud patterns across different transaction types. The main steps are as follows:

1. **Load cleaned data from Task I**  
   The preprocessed dataset saved in Task I (`1_cleaned_fraud_data.csv`) is loaded. Basic properties such as shape and column names are verified to ensure continuity between tasks.

2. **Histogram plots for feature distributions**  
   Histograms are generated for all anonymized transaction features (V1–V28) to visualize their individual distributions. Each histogram includes a kernel density estimate (KDE) overlay for smoother distribution representation. Additionally, the `Amount` feature is visualized both in linear and log10 scales to handle its wide range and skewness.

3. **Violin plots for class comparison**  
   To manage computational overhead and improve plot clarity, stratified sampling is applied (2000 samples per class). Violin plots are created for all V features and the `Amount` feature, showing distribution differences between normal (Class 0) and fraudulent (Class 1) transactions. These plots combine box plots and density curves to reveal central tendency, spread, and distributional shape differences across classes.

4. **Scatter plots for key features (V13, V15, V26)**  
   Individual scatter plots are created for V13, V15, and V26, color-coded by class, to visualize how these features separate fraud from normal transactions across the sample index. Additionally, pairwise scatter plots (V13 vs V15, V13 vs V26, V15 vs V26) are generated to explore potential two-dimensional relationships and clustering patterns between these features.

5. **Fraud analysis: Deposits vs Withdrawals**  
   For features V13, V15, and V26, transactions are categorized as deposits (positive values) or withdrawals (negative values). The number and percentage of fraudulent transactions in each category are calculated and compared using bar plots. This analysis reveals whether fraud is more prevalent in deposit or withdrawal activities, providing actionable insights into fraud behavior patterns.

All visualizations are saved to the `ResultFiles/Task2 Plots/` directory as high-resolution PNG files for inclusion in the final report and further analysis.

In [10]:
# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.figsize'] = (16, 10)

# ----------------------------------------------------------------------------
# 1. Load Cleaned Data from Task I
# ----------------------------------------------------------------------------
print("="*80)
print("TASK II: DATA VISUALIZATION")
print("="*80)
print("\n[1] Loading cleaned dataset from Task I:")

# Load the cleaned data
data_path = 'ResultFiles/1_cleaned_fraud_data.csv'
df = pd.read_csv(data_path)
print(f"    Shape: {df.shape}")
print(f"    Columns: {list(df.columns)}")

# Create output directory for plots
task2_output_dir = "ResultFiles/Task2 Plots"
os.makedirs(task2_output_dir, exist_ok=True)
print(f">>> Task2 result directory: {task2_output_dir}")

TASK II: DATA VISUALIZATION

[1] Loading cleaned dataset from Task I:
    Shape: (99868, 31)
    Columns: ['id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
>>> Task2 result directory: ResultFiles/Task2 Plots


In [11]:
# ----------------------------------------------------------------------------
# 2. HISTOGRAM PLOTS
# ----------------------------------------------------------------------------
print("\n[2] Creating Histogram plots:")

# 2.1 Histogram for all V features ------------------------------------------------------------
fig = plt.figure(figsize=(20, 24))
gs = GridSpec(7, 4, figure=fig, hspace=0.4, wspace=0.3)
v_columns = [f'V{i}' for i in range(1, 29)]

for idx, col in enumerate(v_columns):
    ax = fig.add_subplot(gs[idx // 4, idx % 4])
    
    # Plot histogram and Add KDE line
    ax.hist(df[col], bins=50, alpha=0.6, color='steelblue', edgecolor='black', density=True)
    df[col].plot(kind='kde', ax=ax, color='darkred', linewidth=2)
    
    # Distribution for each transaction
    ax.set_title(f'Distribution of {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.grid(True, alpha=0.3)

plt.suptitle('Histogram of All Transaction Features (V1-V28)', fontsize=16, fontweight='bold', y=0.995)
plt.savefig(f"{task2_output_dir}/01_histogram_all_features.png", dpi=300, bbox_inches='tight')
print(f"    ✓ Saved: 01_histogram_all_features.png")
plt.close()

# 2.2 Histogram for Amount feature ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Amount distribution (all data)
axes[0].hist(df['Amount'], bins=100, alpha=0.7, color='green', edgecolor='black')
axes[0].set_title('Distribution of Account Balance (Amount)', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# Filter out zero/negative amounts for log scale (better visibility)
amount_positive = df[df['Amount'] > 0]['Amount']
axes[1].hist(np.log10(amount_positive), bins=100, alpha=0.7, color='orange', edgecolor='black')
axes[1].set_title('Distribution of Amount (Log10 Scale)', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Log10(Amount)')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{task2_output_dir}/02_histogram_amount.png", dpi=300, bbox_inches='tight')
print(f"    ✓ Saved: 02_histogram_amount.png")
plt.close()


[2] Creating Histogram plots:
    ✓ Saved: 01_histogram_all_features.png
    ✓ Saved: 02_histogram_amount.png


In [12]:
# ----------------------------------------------------------------------------
# 3. VIOLIN PLOTS
# ----------------------------------------------------------------------------
print("\n[3] Creating Violin plots (using stratified sampling):")

# Sample data for violin plot (to avoid computational overhead)
n_sample_per_class = 2000  # 2000 from each class (running on part of both classes)

df_normal = df[df['Class'] == 0].sample(n=min(n_sample_per_class, df[df['Class'] == 0].shape[0]), random_state=42)
df_fraud = df[df['Class'] == 1].sample(n=min(n_sample_per_class, df[df['Class'] == 1].shape[0]), random_state=42)
df_sampled = pd.concat([df_normal, df_fraud], ignore_index=True)

print(f"    Sampled data size: {df_sampled.shape[0]} rows")
print(f"    - Normal (Class 0): {(df_sampled['Class'] == 0).sum()}")
print(f"    - Fraud (Class 1): {(df_sampled['Class'] == 1).sum()}")

# 3.1 Violin plots for all V features -----------------------------------------------------
fig = plt.figure(figsize=(24, 28))
gs = GridSpec(7, 4, figure=fig, hspace=0.5, wspace=0.4)

for idx, col in enumerate(v_columns):
    ax = fig.add_subplot(gs[idx // 4, idx % 4])
    sns.violinplot(data=df_sampled, x='Class', y=col, hue='Class', palette=['blue', 'red'], ax=ax, inner='quartile', cut=0)
    
    ax.set_title(f'Violin Plot: {col}', fontweight='bold')
    ax.set_xlabel('Class (0=Normal, 1=Fraud)')
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Violin Plots: Feature Distribution by Class (Sampled Data)', fontsize=18, fontweight='bold', y=0.995)
plt.savefig(f"{task2_output_dir}/03_violin_all_features.png", dpi=300, bbox_inches='tight')
print(f"    ✓ Saved: 03_violin_all_features.png")
plt.close()

# 3.2 Violin plot for Amount --------------------------------------------------------------
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
sns.violinplot(data=df_sampled, x='Class', y='Amount', hue='Class', palette=['blue', 'red'], ax=ax, inner='box', cut=0)

ax.set_title('Violin Plot: Account Balance (Amount) by Class', fontweight='bold', fontsize=16)
ax.set_xlabel('Class (0=Normal, 1=Fraud)', fontsize=14)
ax.set_ylabel('Amount', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{task2_output_dir}/04_violin_amount.png", dpi=300, bbox_inches='tight')
print(f"    ✓ Saved: 04_violin_amount.png")
plt.close()


[3] Creating Violin plots (using stratified sampling):
    Sampled data size: 2473 rows
    - Normal (Class 0): 2000
    - Fraud (Class 1): 473
    ✓ Saved: 03_violin_all_features.png
    ✓ Saved: 04_violin_amount.png


In [13]:
# ----------------------------------------------------------------------------
# 4. SCATTER PLOTS FOR V13, V15, V26
# ----------------------------------------------------------------------------
print("\n[4] Creating Scatter plots for V15, V26, V13:")

# Use sampled features for scatter plots as well (for clarity)
scatter_features = ['V13', 'V15', 'V26']

# 4.1 Individual scatter plots ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for idx, feature in enumerate(scatter_features):
    normal_mask = df_sampled['Class'] == 0  # plot Normal class first
    fraud_mask = df_sampled['Class'] == 1   # so fraud points are on top
    
    axes[idx].scatter(df_sampled[normal_mask].index, df_sampled[normal_mask][feature], c='blue', alpha=0.5, s=20, label='Normal (0)', edgecolors='none')
    axes[idx].scatter(df_sampled[fraud_mask].index, df_sampled[fraud_mask][feature], c='red', alpha=0.7, s=30, label='Fraud (1)', edgecolors='black', linewidths=0.5)
    
    axes[idx].set_title(f'Scatter Plot: {feature} by Class', fontweight='bold', fontsize=14)
    axes[idx].set_xlabel('Sample Index', fontsize=12)
    axes[idx].set_ylabel(feature, fontsize=12)
    axes[idx].legend(loc='best')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Transaction Scatter Plots: V13, V15, V26 (Color-coded by Class)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{task2_output_dir}/05_scatter_v13_v15_v26.png", dpi=300, bbox_inches='tight')
print(f"    ✓ Saved: 05_scatter_v13_v15_v26.png")
plt.close()

# 4.2 Pairwise scatter plots (V13 vs V15, V13 vs V26, V15 vs V26) -------------------------
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
pairs = [('V13', 'V15'), ('V13', 'V26'), ('V15', 'V26')]

for idx, (feature1, feature2) in enumerate(pairs):
    normal_mask = df_sampled['Class'] == 0
    fraud_mask = df_sampled['Class'] == 1
    
    axes[idx].scatter(df_sampled[normal_mask][feature1], df_sampled[normal_mask][feature2], c='blue', alpha=0.4, s=20, label='Normal (0)', edgecolors='none')
    axes[idx].scatter(df_sampled[fraud_mask][feature1], df_sampled[fraud_mask][feature2], c='red', alpha=0.7, s=40, label='Fraud (1)', edgecolors='black', linewidths=0.5)
    
    axes[idx].set_title(f'{feature1} vs {feature2}', fontweight='bold', fontsize=14)
    axes[idx].set_xlabel(feature1, fontsize=12)
    axes[idx].set_ylabel(feature2, fontsize=12)
    axes[idx].legend(loc='best')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Pairwise Scatter Plots: V13, V15, V26 (Color-coded by Class)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{task2_output_dir}/06_scatter_pairwise.png", dpi=300, bbox_inches='tight')
print(f"    ✓ Saved: 06_scatter_pairwise.png")
plt.close()


[4] Creating Scatter plots for V15, V26, V13:
    ✓ Saved: 05_scatter_v13_v15_v26.png
    ✓ Saved: 06_scatter_pairwise.png


In [14]:
# ----------------------------------------------------------------------------
# 5. ANALYSIS & CONCLUSION: DEPOSIT VS WITHDRAWAL FRAUD
# ----------------------------------------------------------------------------
print("\n[5] Analyzing fraud in deposits vs withdrawals:")

# Separate positive (deposits) and negative (withdrawals) transactions
fraud_data = df[df['Class'] == 1]

# For V13, V15, V26: positive = deposit, negative = withdrawal
features_to_analyze = ['V13', 'V15', 'V26']
analysis_results = {}

for feature in features_to_analyze:
    deposits = fraud_data[fraud_data[feature] > 0]
    withdrawals = fraud_data[fraud_data[feature] < 0]
    
    deposit_count = len(deposits)
    withdrawal_count = len(withdrawals)
    total_fraud = len(fraud_data)
    
    analysis_results[feature] = {
        'deposits': deposit_count,
        'withdrawals': withdrawal_count,
        'deposit_pct': (deposit_count / total_fraud) * 100 if total_fraud > 0 else 0,
        'withdrawal_pct': (withdrawal_count / total_fraud) * 100 if total_fraud > 0 else 0
    }
    
    print(f"\n    {feature}:")
    print(f"      Fraud in Deposits: {deposit_count} ({analysis_results[feature]['deposit_pct']:.2f}%)")
    print(f"      Fraud in Withdrawals: {withdrawal_count} ({analysis_results[feature]['withdrawal_pct']:.2f}%)")

# 5.1 Bar plot comparing deposits vs withdrawals ------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Count comparison
categories = features_to_analyze
deposits_counts = [analysis_results[f]['deposits'] for f in features_to_analyze]
withdrawals_counts = [analysis_results[f]['withdrawals'] for f in features_to_analyze]

x = np.arange(len(categories))
width = 0.35

axes[0].bar(x - width/2, deposits_counts, width, label='Deposits', color='green', alpha=0.7, edgecolor='black')
axes[0].bar(x + width/2, withdrawals_counts, width, label='Withdrawals', color='red', alpha=0.7, edgecolor='black')
axes[0].set_title('Fraud Count: Deposits vs Withdrawals', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Transaction Feature', fontsize=12)
axes[0].set_ylabel('Fraud Count', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Percentage comparison
deposits_pcts = [analysis_results[f]['deposit_pct'] for f in features_to_analyze]
withdrawals_pcts = [analysis_results[f]['withdrawal_pct'] for f in features_to_analyze]

axes[1].bar(x - width/2, deposits_pcts, width, label='Deposits', color='green', alpha=0.7, edgecolor='black')
axes[1].bar(x + width/2, withdrawals_pcts, width, label='Withdrawals', color='red', alpha=0.7, edgecolor='black')
axes[1].set_title('Fraud Percentage: Deposits vs Withdrawals', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Transaction Feature', fontsize=12)
axes[1].set_ylabel('Fraud Percentage (%)', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{task2_output_dir}/07_fraud_deposit_vs_withdrawal.png", dpi=300, bbox_inches='tight')
print(f"\n    ✓ Saved: 07_fraud_deposit_vs_withdrawal.png")
plt.close()

# 5.1 Conclusion --------------------------------------------------------------------------
print("\n" + "="*80)
print("TASK II COMPLETED SUCCESSFULLY")
print("="*80)
print(f"\n    All plots saved to: {task2_output_dir}/")
print("    Generated files:")
print("    1. 01_histogram_all_features.png      - Histograms of V1-V28")
print("    2. 02_histogram_amount.png            - Amount distribution")
print("    3. 03_violin_all_features.png         - Violin plots for all features")
print("    4. 04_violin_amount.png               - Violin plot for Amount")
print("    5. 05_scatter_v13_v15_v26.png         - Individual scatter plots")
print("    6. 06_scatter_pairwise.png            - Pairwise scatter plots")
print("    7. 07_fraud_deposit_vs_withdrawal.png - Deposit vs Withdrawal analysis")
print("\n" + "="*80)


[5] Analyzing fraud in deposits vs withdrawals:

    V13:
      Fraud in Deposits: 234 (49.47%)
      Fraud in Withdrawals: 239 (50.53%)

    V15:
      Fraud in Deposits: 226 (47.78%)
      Fraud in Withdrawals: 247 (52.22%)

    V26:
      Fraud in Deposits: 241 (50.95%)
      Fraud in Withdrawals: 232 (49.05%)

    ✓ Saved: 07_fraud_deposit_vs_withdrawal.png

TASK II COMPLETED SUCCESSFULLY

    All plots saved to: ResultFiles/Task2 Plots/
    Generated files:
    1. 01_histogram_all_features.png      - Histograms of V1-V28
    2. 02_histogram_amount.png            - Amount distribution
    3. 03_violin_all_features.png         - Violin plots for all features
    4. 04_violin_amount.png               - Violin plot for Amount
    5. 05_scatter_v13_v15_v26.png         - Individual scatter plots
    6. 06_scatter_pairwise.png            - Pairwise scatter plots
    7. 07_fraud_deposit_vs_withdrawal.png - Deposit vs Withdrawal analysis



## TASK III: Fraud Detection Classification

In this task, we develop and compare machine learning models for fraud detection using the cleaned dataset from Task I and insights from Task II. The workflow includes feature engineering, feature selection, model training with class imbalance handling, and comprehensive evaluation. The main steps are as follows:

1. **Load cleaned data and assess class imbalance**  
   The preprocessed dataset (`1_cleaned_fraud_data.csv`) is loaded. Class distribution reveals a severe imbalance ratio of approximately 210:1 (normal vs fraud transactions), confirming the need for specialized handling techniques.

2. **Feature engineering**  
   Five new features are created to enhance model discriminative power:
   - `Amount_log`: Log transformation of transaction amount to handle right-skewed distribution
   - `is_withdrawal`: Binary indicator based on V15 < 0 (from Task II insights showing fraud prevalence in withdrawals)
   - `V15_V17`, `V15_V26`, `V17_V26`: Interaction features combining top discriminative variables identified in Task II

3. **Feature selection using Mutual Information (MI)**  
   Mutual Information scores are calculated for all features (original V1-V28 plus engineered features, excluding `Amount` to avoid data leakage). The top 20 features with highest MI scores are selected, including engineered features `V15_V17` (MI=0.0164, rank 4), `is_withdrawal` (MI=0.0154, rank 5), and `V17_V26` (MI=0.0116, rank 9). This statistical approach ensures features with strongest non-linear relationships to fraud are retained.

4. **Train-test split and feature scaling**  
   Data is split 80-20 with stratification to preserve class distribution in both sets (train: 79,494 samples; test: 19,874 samples). StandardScaler is applied to ensure features have mean=0 and std=1, critical for distance-based algorithms like Logistic Regression.

5. **Model training with class imbalance handling**  
   Two models are trained with `class_weight='balanced'` to automatically assign higher penalties to minority class (fraud) misclassifications:
   - **Logistic Regression**: Linear baseline model with C=0.1 regularization
   - **Random Forest**: Ensemble model with 100 trees, max_depth=10 to prevent overfitting

6. **Model evaluation with fraud-focused metrics**  
   Both models are evaluated using a **fixed threshold of 0.5** (no optimization) to ensure fair comparison. Metrics include:
   - **Precision**: Proportion of predicted frauds that are actual frauds
   - **Recall**: Proportion of actual frauds correctly detected (critical for fraud detection)
   - **F1-Score**: Harmonic mean of Precision and Recall
   - **F2-Score**: Weighted F-score with β=2, emphasizing Recall over Precision

7. **Results and visualization**  
   - **Confusion Matrices** (`10_confusion_matrices.png`): Visual comparison showing Random Forest achieves better balance between True Positives and False Positives
   - **Feature Importance** (`11_feature_importance.png`): Top 15 features from Random Forest reveal that engineered features (`V15_V17`, `is_withdrawal`) rank highly alongside original features (V17, V14, V12)
   - **Model Comparison** (`12_model_comparison.csv`): Summary table showing:
     * **Logistic Regression**: Precision=0.20, Recall=0.87, F1=0.33, F2=0.52
     * **Random Forest**: Precision=0.87, Recall=0.79, F1=0.83, F2=0.80

8. **Best model selection**  
   **Random Forest** is selected as the best model based on F1-Score (0.83), achieving superior balance between precision and recall. While Logistic Regression has higher recall (0.87), its extremely low precision (0.20) results in excessive false alarms, making it impractical for deployment.

All results, including confusion matrices, feature importance plot, and model comparison summary, are saved to `ResultFiles/Task3 Classification/` for inclusion in the final report.

In [15]:
# Style settings (consistent with Task II)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("TASK III: FRAUD DETECTION CLASSIFICATION")
print("="*80)

# ----------------------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------------------
print("\n[1] Loading dataset:")
data_path = 'ResultFiles/1_cleaned_fraud_data.csv'
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:")
print(f"{df['Class'].value_counts()}")
print(f"Imbalance ratio: {df['Class'].value_counts()[0] / df['Class'].value_counts()[1]:.1f}:1")

# ----------------------------------------------------------------------------
# 2. EXPLORATORY DATA ANALYSIS
# ----------------------------------------------------------------------------
print("\n[2] Quick EDA:")
print(f"Missing values: {df.isnull().sum().sum()}")
print("Key features statistics:")
print(df[['Amount', 'V15', 'V17', 'V26', 'Class']].describe())

# Create output directory for plots
task3_output_dir = "ResultFiles/Task3 Classification"
os.makedirs(task3_output_dir, exist_ok=True)
print(f"\n>>> Task3 result directory: {task3_output_dir}")

TASK III: FRAUD DETECTION CLASSIFICATION

[1] Loading dataset:
Dataset shape: (99868, 31)
Class distribution:
Class
0    99395
1      473
Name: count, dtype: int64
Imbalance ratio: 210.1:1

[2] Quick EDA:
Missing values: 0
Key features statistics:
             Amount           V15           V17           V26         Class
count  99868.000000  99868.000000  99868.000000  99868.000000  99868.000000
mean      98.220912      0.193517      0.044867      0.027068      0.004736
std      265.007039      0.929739      1.022641      0.494731      0.068658
min        0.000000     -4.498945    -25.162799     -2.534330      0.000000
25%        7.547500     -0.357326     -0.403304     -0.324602      0.000000
50%       26.490000      0.300280     -0.002949     -0.069743      0.000000
75%       89.310000      0.865571      0.453933      0.300675      0.000000
max    19656.530000      5.784514      9.253526      3.517346      1.000000

>>> Task3 result directory: ResultFiles/Task3 Classification


In [16]:
# ----------------------------------------------------------------------------
# 3. FEATURE ENGINEERING
# ----------------------------------------------------------------------------
print("\n[3] Feature Engineering:")

# 3.1 Log transformation for Amount (highly right-skewed) ---------------------------------
df['Amount_log'] = np.log1p(df['Amount'])
print("    - Amount_log: log(1 + Amount) transformation")

# 3.2 Binary feature: withdrawal vs deposit (based on Task II analysis) -------------------
df['is_withdrawal'] = (df['V15'] < 0).astype(int)
print("    - is_withdrawal: binary feature based on V15 < 0")

# 3.3 Interaction features (top discriminative features from Task II) ---------------------
df['V15_V17'] = df['V15'] * df['V17']
df['V15_V26'] = df['V15'] * df['V26']
df['V17_V26'] = df['V17'] * df['V26']  
print("    - V15_V17, V15_V26, V17_V26: interaction features")

print(f"    Total features created: 5") 

# ----------------------------------------------------------------------------
# 4. FEATURE SELECTION (MUTUAL INFORMATION)
# ----------------------------------------------------------------------------
print("\n[4] Feature Selection using Mutual Information:")

# Prepare feature matrix (exclude id, Class, original Amount) 
feature_cols = [col for col in df.columns if col not in ['id', 'Class', 'Amount']]
X = df[feature_cols]
y = df['Class']

# Calculate Mutual Information scores 
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({
    'Feature': feature_cols,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

# Select top 20 features 
top_n = 20
selected_features = mi_df.head(top_n)['Feature'].tolist()
print(f"    Selected top {top_n} features:")
print(mi_df.head(top_n).to_string(index=False))

X_selected = X[selected_features]


[3] Feature Engineering:
    - Amount_log: log(1 + Amount) transformation
    - is_withdrawal: binary feature based on V15 < 0
    - V15_V17, V15_V26, V17_V26: interaction features
    Total features created: 5

[4] Feature Selection using Mutual Information:
    Selected top 20 features:
      Feature  MI_Score
          V14  0.019932
          V17  0.019678
          V10  0.018206
          V12  0.017572
          V11  0.015255
           V3  0.014619
          V16  0.014343
           V4  0.013392
      V17_V26  0.011594
      V15_V17  0.010411
           V9  0.009934
           V7  0.009843
          V18  0.009385
is_withdrawal  0.008762
           V2  0.008433
          V21  0.006623
          V27  0.006280
           V6  0.006036
           V1  0.005588
           V5  0.005579


In [17]:
# ----------------------------------------------------------------------------
# 5. TRAIN-TEST SPLIT
# ----------------------------------------------------------------------------
print("\n[5] Splitting data (80-20 with stratification):")
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42, stratify=y)
print(f"    Train set: {X_train.shape}")
print(f"    Test set:  {X_test.shape}")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

# ----------------------------------------------------------------------------
# 6. FEATURE SCALING
# ----------------------------------------------------------------------------
print("\n[6] Feature scaling with StandardScaler:")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("    Features standardized (mean=0, std=1)")


[5] Splitting data (80-20 with stratification):
    Train set: (79894, 20)
    Test set:  (19974, 20)

Train class distribution:
Class
0    79516
1      378
Name: count, dtype: int64

Test class distribution:
Class
0    19879
1       95
Name: count, dtype: int64

[6] Feature scaling with StandardScaler:
    Features standardized (mean=0, std=1)


In [18]:
# ----------------------------------------------------------------------------
# 7. MODEL TRAINING
# ----------------------------------------------------------------------------
print("\n[7] Training models with class_weight='balanced':")

# Model 1: Logistic Regression ------------------------------------------------------------
print("    [1/2] Logistic Regression:")
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=0.1,
    random_state=42
)
lr_model.fit(X_train_scaled, y_train)
print("        Training completed")

# Model 2: Random Forest ------------------------------------------------------------------
print("    [2/2] Random Forest:")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
print("        Training completed")


[7] Training models with class_weight='balanced':
    [1/2] Logistic Regression:
        Training completed
    [2/2] Random Forest:
        Training completed


In [19]:
# ----------------------------------------------------------------------------
# 8. MODEL EVALUATION
# ----------------------------------------------------------------------------
print("\n[8] Model evaluation:")

results = {}

for model_name, model in [('Logistic Regression', lr_model), ('Random Forest', rf_model)]:
    print(f"\n{'='*80}")
    print(f"{model_name}")
    print('='*80)
    
    # Predictions with default threshold (0.5)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    
    # Calculate metrics (Lecture 8: Precision, Recall, F-beta)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    f2 = fbeta_score(y_test, y_pred, beta=2)  # beta=2 emphasizes Recall
    
    # Store results
    results[model_name] = {
        'y_pred': y_pred,
        'y_proba': y_proba,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f2': f2
    }
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"F2-Score:  {f2:.4f} (beta=2, emphasizes Recall)")
    
    # Classification report
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Fraud (1)']))


[8] Model evaluation:

Logistic Regression
Precision: 0.1907
Recall:    0.8632
F1-Score:  0.3124
F2-Score:  0.5062 (beta=2, emphasizes Recall)

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       1.00      0.98      0.99     19879
   Fraud (1)       0.19      0.86      0.31        95

    accuracy                           0.98     19974
   macro avg       0.60      0.92      0.65     19974
weighted avg       1.00      0.98      0.99     19974


Random Forest
Precision: 0.8706
Recall:    0.7789
F1-Score:  0.8222
F2-Score:  0.7957 (beta=2, emphasizes Recall)

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00     19879
   Fraud (1)       0.87      0.78      0.82        95

    accuracy                           1.00     19974
   macro avg       0.93      0.89      0.91     19974
weighted avg       1.00      1.00      1.00     19974



In [20]:
# ----------------------------------------------------------------------------
# 9. CONFUSION MATRICES (MANDATORY)
# ----------------------------------------------------------------------------
print("\n[9] Generating Confusion Matrices:")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (model_name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result['y_pred'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Fraud'],
                yticklabels=['Normal', 'Fraud'],
                ax=axes[idx], cbar_kws={'label': 'Count'})
    
    axes[idx].set_title(f'{model_name}\nConfusion Matrix', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=12)
    axes[idx].set_ylabel('Actual', fontsize=12)

plt.tight_layout()
plt.savefig(f'{task3_output_dir}/10_confusion_matrices.png', dpi=300, bbox_inches='tight')
print("    ✓ Saved: 10_confusion_matrices.png")
plt.close()


[9] Generating Confusion Matrices:
    ✓ Saved: 10_confusion_matrices.png


In [21]:
# ----------------------------------------------------------------------------
# 10. FEATURE IMPORTANCE (FOR EXPLANATION)
# ----------------------------------------------------------------------------
print("\n[10] Analyzing Feature Importance:")

# Extract feature importance from Random Forest
importances = rf_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': selected_features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
top_15 = importance_df.head(15)
plt.barh(range(len(top_15)), top_15['Importance'], color='steelblue')
plt.yticks(range(len(top_15)), top_15['Feature'])
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 15 Feature Importance (Random Forest)', fontsize=14, fontweight='bold')

plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{task3_output_dir}/11_feature_importance.png', dpi=300, bbox_inches='tight')
print("    ✓ Saved: 11_feature_importance.png")
print(f"\n    Top 10 features:")
print(importance_df.head(10).to_string(index=False))
plt.close()


[10] Analyzing Feature Importance:
    ✓ Saved: 11_feature_importance.png

    Top 10 features:
Feature  Importance
    V14    0.226228
     V3    0.156482
    V12    0.123671
    V10    0.104872
    V17    0.083728
     V4    0.078938
    V16    0.043343
    V11    0.037316
     V9    0.019895
     V2    0.019296


In [22]:
# ----------------------------------------------------------------------------
# 11. SUMMARY RESULTS
# ----------------------------------------------------------------------------
print("\n[11] Summary of Results:")

summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Precision': [results[m]['precision'] for m in results.keys()],
    'Recall': [results[m]['recall'] for m in results.keys()],
    'F1-Score': [results[m]['f1'] for m in results.keys()],
    'F2-Score': [results[m]['f2'] for m in results.keys()]
})
print(f"\n{summary.to_string(index=False)}")

# Save summary to CSV
summary.to_csv(f'{task3_output_dir}/12_model_comparison.csv', index=False)
print("\n    ✓ Saved: 12_model_comparison.csv")

# ----------------------------------------------------------------------------
# 12. BEST MODEL SELECTION
# ----------------------------------------------------------------------------
best_model_name = summary.loc[summary['F1-Score'].idxmax(), 'Model']
print(f"\n[12] Best Model (based on F1-Score): {best_model_name}")

print("\n    Generated Files:")
print("    1. 10_confusion_matrices.png   - Confusion matrices for both models")
print("    2. 11_feature_importance.png   - Top 15 feature importance")
print("    3. 12_model_comparison.csv     - Model comparison summary")

print("\n" + "="*80)
print("TASK III COMPLETED SUCCESSFULLY!")
print("="*80)


[11] Summary of Results:

              Model  Precision   Recall  F1-Score  F2-Score
Logistic Regression   0.190698 0.863158  0.312381  0.506173
      Random Forest   0.870588 0.778947  0.822222  0.795699

    ✓ Saved: 12_model_comparison.csv

[12] Best Model (based on F1-Score): Random Forest

    Generated Files:
    1. 10_confusion_matrices.png   - Confusion matrices for both models
    2. 11_feature_importance.png   - Top 15 feature importance
    3. 12_model_comparison.csv     - Model comparison summary

TASK III COMPLETED SUCCESSFULLY!
